In [3]:
import glob
import io
import numpy as np
import os
import pandas as pd
from PIL import Image

IMAGE_CLASS_DIR = "/data/susanket/vlm/eval/image-class"  # Path to folder containing class subdirectories
IMAGE_CAP_DIR = "/data/susanket/vlm/eval/image-cap"      # Path to folder containing existing parquet files
OUTPUT_PARQUET = "/data/susanket/vlm/sft/test.parquet"

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff", ".tif"}

In [14]:
def extract_raw_bytes(val) -> bytes:
    """Helper to safely extract raw byte stream whether input is bytes, dict, or HF format."""
    if isinstance(val, bytes):
        return val
    if isinstance(val, dict) and "bytes" in val:
        return val["bytes"]
    if hasattr(val, "get") and val.get("bytes") is not None:
        return val.get("bytes")
    return bytes(val) if val is not None else b""

def process_image_class(class_root: str) -> list:
    """
    Scans nested image-class directory layout:
    image-class/ -> dataset_name/ -> class_name/ -> image.jpg
    Reads image files directly as raw bytes into image_bytes column.
    """
    records = []
    if not os.path.exists(class_root):
        print(f"Directory not found: {class_root}")
        return records

    print("Processing image-class directory...")
    
    # Level 1: Iterate over datasets
    for ds_name in os.listdir(class_root):
        ds_dir = os.path.join(class_root, ds_name)
        if not os.path.isdir(ds_dir):
            continue

        # Level 2: Iterate over classes
        for class_name in os.listdir(ds_dir):
            class_dir = os.path.join(ds_dir, class_name)
            if not os.path.isdir(class_dir):
                continue

            caption_text = class_name.replace("_", " ").strip()

            # Level 3: Read image files as bytes
            for filename in os.listdir(class_dir):
                ext = os.path.splitext(filename)[1].lower()
                if ext in VALID_EXTENSIONS:
                    img_path = os.path.join(class_dir, filename)
                    
                    try:
                        with open(img_path, "rb") as f:
                            img_bytes = f.read()
                            
                        records.append({
                            "dataset": ds_name,
                            "filename": filename,
                            "caption": caption_text,
                            "dataset_type": "class",
                            "image_bytes": img_bytes
                        })
                    except Exception as e:
                        print(f"Error reading {img_path}: {e}")

    print(f"Collected {len(records)} samples from image-class.")
    return records


def process_image_cap(cap_root: str) -> list:
    """
    Reads existing Parquet files in image-cap directory, safely extracts 
    image bytes (handling dict formats), and stores them under image_bytes.
    """
    records = []
    if not os.path.exists(cap_root):
        print(f"Directory not found: {cap_root}")
        return records

    print("Processing image-cap directory...")
    parquet_files = glob.glob(os.path.join(cap_root, "*.parquet"))

    for p_file in parquet_files:
        ds_name = os.path.splitext(os.path.basename(p_file))[0]
        print(f"Reading {p_file}...")
        df = pd.read_parquet(p_file)

        # Look up image and caption columns dynamically
        image_col = None
        for col in ["image_bytes", "image_byte", "image", "img_bytes"]:
            if col in df.columns:
                image_col = col
                break

        caption_col = None
        for col in ["caption", "captions", "raw"]:
            if col in df.columns:
                caption_col = col
                break

        if not image_col or not caption_col:
            print(f"Skipping {p_file}: required columns missing.")
            continue

        for _, row in df.iterrows():
            # Safely extract binary bytes from column value
            img_bytes = extract_raw_bytes(row[image_col])
            caption = row[caption_col]

            # Handle list/array captions
            if isinstance(caption, (list, tuple, range, np.ndarray)):
                caption = caption[0] if len(caption) > 0 else ""
            elif not isinstance(caption, str):
                caption = str(caption)

            records.append({
                "dataset": ds_name,
                "filename": row.get("filename", f"{ds_name}_sample"),
                "caption": caption,
                "dataset_type": "caption",
                "image_bytes": img_bytes
            })

    print(f"Collected {len(records)} samples from image-cap.")
    return records


def build_unified_parquet():
    all_records = []

    # 1. Process image-class
    class_records = process_image_class(IMAGE_CLASS_DIR)
    all_records.extend(class_records)

    # 2. Process image-cap
    cap_records = process_image_cap(IMAGE_CAP_DIR)
    all_records.extend(cap_records)

    if not all_records:
        print("No records found in either directory.")
        return

    # 3. Combine into single DataFrame
    df = pd.DataFrame(all_records)

    # 4. Export using PyArrow binary serialization
    df.to_parquet(OUTPUT_PARQUET, engine="pyarrow", index=False)
    print(f"\nSuccessfully written {len(df)} total records to '{OUTPUT_PARQUET}'!")


if __name__ == "__main__":
    build_unified_parquet()

Processing image-class directory...


Collected 63910 samples from image-class.
Processing image-cap directory...
Reading /data/susanket/vlm/eval/image-cap/UCM.parquet...
Reading /data/susanket/vlm/eval/image-cap/RSICD.parquet...
Reading /data/susanket/vlm/eval/image-cap/RSITMD.parquet...
Collected 17764 samples from image-cap.

Successfully written 81674 total records to '/data/susanket/vlm/sft/test.parquet'!


In [4]:
import pandas as pd
import numpy as np

In [18]:
df = pd.read_parquet(rf"/data/susanket/vlm/sft/test.parquet")

In [20]:
df

,dataset,filename,caption,dataset_type,image_bytes,translated_col,translated_lang
0,RSICB128,sand_beach(11).tif,sandbeach,class,b'II*\x00\x8c\xc1\x00\x00\x7f\xba\xc1\xc9\xc4\...,沙灘,Cantonese
1,RSICB128,sand_beach(382).tif,sandbeach,class,b'II*\x00j\xc1\x00\x00\x7f\xea\xed\xe6\xea\xed...,沙,Simplified Chinese
2,RSICB128,sand_beach(516).tif,sandbeach,class,b'II*\x00\x88\xc1\x00\x00\x7f\xe4\xe6\xdb\xe3\...,沙,Simplified Chinese
3,RSICB128,sand_beach(317).tif,sandbeach,class,b'II*\x00f\xc1\x00\x00\x7f\xeb\xe8\xe3\xeb\xe8...,plage de sable,French
4,RSICB128,sand_beach(105).tif,sandbeach,class,b'II*\x00\x88\xc1\x00\x00\x7f\xb2\xba\xbd\xb1\...,playa de arena,Spanish
...,...,...,...,...,...,...,...
81669,RSITMD,denseresidential_447.tif,"There is a major crossroads in the area, with ...",caption,b'II*\x00\x12\x06\x03\x00\x7f\x84\x8f\x89\x84\...,"Il y a un carrefour important dans la région, ...",French
81670,RSITMD,airport_448.tif,The plane was surrounded by some vehicles.,caption,b'II*\x00\x12\x06\x03\x00\x7f\xbf\xd2\xcc\xbd\...,विमान कुछ वाहनों से घिरा हुआ था।,Hindi
81671,RSITMD,intersection_449.tif,It is a medium residential area .,caption,b'II*\x00\x1c\x00\x03\x00Sh]dwpr\x80wz\x88{}zr...,यह एक मध्यम आवासीय क्षेत्र है .,Hindi
81672,RSITMD,playground_450.tif,There are small plant tennis courts.,caption,b'II*\x00\x16\x06\x03\x00\x7fxdgqg`ulg\x95\x93...,Há pequenas quintas de tênis.,Portuguese


In [21]:
import pyarrow.parquet as pq

src = "/data/susanket/vlm/sft/test.parquet"
dst = "/data/susanket/vlm/sft/test2.parquet"

pf = pq.ParquetFile(src)
with pq.ParquetWriter(dst, pf.schema_arrow, compression="zstd", row_group_size=512) as w:
    for batch in pf.iter_batches(batch_size=512):
        w.write_batch(batch)
print("done")


TypeError: __cinit__() got an unexpected keyword argument 'row_group_size'

In [22]:
import os
import pyarrow.parquet as pq

src = "/data/susanket/vlm/sft/test.parquet"
dst = "/data/susanket/vlm/sft/test2.parquet"

if os.path.exists(dst):
    os.remove(dst)   # the failed run may have left an empty/partial file

table = pq.read_table(src)          # ~6.5 GB in memory for test.parquet
pq.write_table(table, dst, row_group_size=512, compression="zstd")
print("rows", table.num_rows, "written to", dst)

rows 81674 written to /data/susanket/vlm/sft/test2.parquet


In [23]:
import pyarrow.parquet as pq
pf = pq.ParquetFile("/data/susanket/vlm/sft/test2.parquet")
print(pf.metadata.num_rows, pf.num_row_groups, pf.metadata.row_group(0).num_rows)

81674 160 512
